# Base oficial de passageiros transportados (SPTrans / Prefeitura de São Paulo)

Baixa e compila o **demonstrativo diário de passageiros transportados por linha de ônibus, por
modalidade de pagamento**, para todos os dias de 2023 e 2024.

É a única referência **externa e auditável** de demanda do projeto: a bilhetagem vem de pedido
e-SIC e não pode ser conferida contra nada; esta série é publicada. Serve para validar os níveis
da bilhetagem e para reportar totais de linha/cidade.

Fonte: `prefeitura.sp.gov.br/mobilidade/w/institucional/sptrans/acesso_a_informacao/152416`
(páginas anuais em `/passageiros-transportados-<ano>`), um arquivo por dia.

Saída: `../outputs/01/Dados_4Meses_Domingos_2023_2024_site.parquet`.

> **Nota sobre o nome do arquivo.** O nome espelha o irmão de bilhetagem
> (`Dados_4Meses_Domingos_2023_2024_genero_idade`), mas o conteúdo cobre **todos os dias** de 2023
> e 2024, não só os 4 meses e os domingos. O recorte de análise é um filtro sobre esta base.

## Seção 0 — Constantes

In [1]:
import datetime
import glob
import os
import re
import time
import unicodedata
import urllib.parse

import pandas as pd
import requests

# Raiz das fontes brutas (fora do repo). Se a unidade mudar, muda so esta linha.
RAIZ_SPTRANS = r"C:\Users\9837292\Desktop\SSD\SPTrans"

# Onde ficam os .xls/.xlsx crus
PASTA_OFICIAL = os.path.join(RAIZ_SPTRANS, "passageiros transportados")

# Convenção do repositório: outputs vão para ../outputs/<NN>/, onde NN é o prefixo da
# pasta do notebook que os gerou (este está em 01_criacao_de_bases/).
PASTA_SAIDA = "../outputs/01"
CAMINHO_SAIDA = f"{PASTA_SAIDA}/Dados_4Meses_Domingos_2023_2024_site.parquet"

ANOS = [2023, 2024]
BASE_UPLOAD = "https://www.prefeitura.sp.gov.br/cidade/secretarias/upload"
PAGINAS_ANUAIS = {
    2023: "https://prefeitura.sp.gov.br/web/mobilidade/w/institucional/sptrans/acesso_a_informacao/343693",
    2024: "https://prefeitura.sp.gov.br/web/mobilidade/w/institucional/sptrans/acesso_a_informacao/362878",
}
MESES_ABR = {1: "JAN", 2: "FEV", 3: "MAR", 4: "ABR", 5: "MAI", 6: "JUN",
             7: "JUL", 8: "AGO", 9: "SET", 10: "OUT", 11: "NOV", 12: "DEZ"}
SUFIXOS = ["(3)", "(2)", "(1)", ""]   # republicações; conteúdo idêntico (conferido em 08JAN2023)

FORCE_REDOWNLOAD = False   # True refaz o download mesmo dos arquivos já presentes
FORCE_RECOMPUTE = False    # True recompila o parquet mesmo se ele já existir

os.makedirs(PASTA_OFICIAL, exist_ok=True)
os.makedirs(PASTA_SAIDA, exist_ok=True)
pd.set_option("display.width", 200)

## Seção 1 — Download

**O portal usa duas famílias de URL**, porque migrou de plataforma no meio de 2024:

1. `.../cidade/secretarias/upload/DDMMMAAAA.xls` — 2023 inteiro e 2024 até ~julho;
2. `/documents/d/mobilidade/DDmmmAAAA-...` — de julho/2024 em diante.

A fonte primária são os `href` das páginas anuais; a URL construída é fallback (necessária porque
a página de 2024 só lista até junho, embora os arquivos de julho/agosto existam no padrão antigo).

Os arquivos `Consolidado MM-Mes-AAAA.xls` são agregados mensais e devem ficar de fora — o
casamento por `\\d{2}[A-Z]{3}\\d{4}` já os exclui, porque o nome deles não começa com a data.

In [2]:
ses = requests.Session()
ses.headers.update({"User-Agent": "Mozilla/5.0 (pesquisa academica CEM/USP)"})


def dias_do_ano(ano):
    d = datetime.date(ano, 1, 1)
    while d.year == ano:
        yield d
        d += datetime.timedelta(days=1)


def indexa_paginas_anuais():
    """(dia, MES, ano) -> [urls], a partir dos hrefs das páginas anuais."""
    idx = {}
    for ano, pagina in PAGINAS_ANUAIS.items():
        html = None
        for tentativa in range(4):
            try:
                r = ses.get(pagina, timeout=240)   # a página é pesada e às vezes dá timeout
                r.raise_for_status()
                html = r.text
                break
            except Exception as e:
                print(f"  retry {ano} ({tentativa + 1}/4): {type(e).__name__}", flush=True)
                time.sleep(5)
        if html is None:
            print(f"  ATENÇÃO: página de {ano} inacessível; só o fallback será usado", flush=True)
            continue
        for href in set(re.findall(r'href=["\']([^"\']+)["\']', html)):
            nome = urllib.parse.unquote(href).rsplit("/", 1)[-1]
            m = re.match(r"(\d{2})([A-Za-z]{3})(\d{4})", nome)
            if not m:
                continue
            chave = (int(m.group(1)), m.group(2).upper(), int(m.group(3)))
            url = href if href.startswith("http") else "https://prefeitura.sp.gov.br" + href
            idx.setdefault(chave, []).append(url.replace("https://https://", "https://"))
    return idx


def candidatos_de(d, idx):
    def rank(u):  # republicação mais recente primeiro
        m = re.search(r"[-(](\d+)[)-]x?ls", u) or re.search(r"\((\d+)\)\.xls", u)
        return int(m.group(1)) if m else 0
    urls = sorted(idx.get((d.day, MESES_ABR[d.month], d.year), []), key=rank, reverse=True)
    return urls + [f"{BASE_UPLOAD}/{d.day:02d}{MESES_ABR[d.month]}{d.year}{s}.xls" for s in SUFIXOS]


def ja_baixado(d):
    return any(os.path.exists(os.path.join(PASTA_OFICIAL, f"{d:%Y%m%d}{e}"))
               and os.path.getsize(os.path.join(PASTA_OFICIAL, f"{d:%Y%m%d}{e}")) > 10000
               for e in (".xls", ".xlsx"))


def baixa_dia(d, idx):
    if not FORCE_REDOWNLOAD and ja_baixado(d):
        return "ja_existe"
    for url in candidatos_de(d, idx):
        try:
            r = ses.get(url, timeout=120)
        except Exception:
            continue
        if r.status_code == 200 and len(r.content) > 10000:
            ext = ".xlsx" if r.content[:2] == b"PK" else ".xls"   # assinatura ZIP => xlsx
            with open(os.path.join(PASTA_OFICIAL, f"{d:%Y%m%d}{ext}"), "wb") as f:
                f.write(r.content)
            return "ok"
    return "falhou"


alvo = [d for ano in ANOS for d in dias_do_ano(ano)]
if all(ja_baixado(d) for d in alvo) and not FORCE_REDOWNLOAD:
    print(f"Todos os {len(alvo)} dias já estão em {PASTA_OFICIAL}.")
else:
    idx = indexa_paginas_anuais()
    print(f"{len(idx)} datas indexadas nos hrefs das páginas anuais", flush=True)
    ok = existia = 0
    faltando = []
    for i, d in enumerate(alvo, 1):
        estado = baixa_dia(d, idx)
        ok += estado == "ok"
        existia += estado == "ja_existe"
        if estado == "falhou":
            faltando.append(str(d))
        if i % 100 == 0:
            print(f"  {i}/{len(alvo)} baixados={ok} existiam={existia} faltam={len(faltando)}", flush=True)
        time.sleep(0.15)
    print(f"\nalvo={len(alvo)} baixados={ok} já_existiam={existia} faltando={len(faltando)}")
    for f in faltando:
        print("   falta:", f)

Todos os 731 dias já estão em C:\Users\9837292\Desktop\SSD\SPTrans\passageiros transportados.


## Seção 2 — Leitura de um arquivo

Quatro armadilhas deste formato, todas verificadas nos arquivos reais:

1. **O cabeçalho não está em linha fixa.** Nos `.xls` de 2023 está na linha 2; nos `.xlsx` de
   ago-out/2024, na linha 3. É localizado procurando a linha que contém `Data` e `Linha`.
2. **`Data` vem ora como texto `dd/mm/aaaa`, ora como `datetime`.** Normalizamos e **conferimos
   contra a data do nome do arquivo** — é a checagem que garante que o arquivo baixado é do dia
   que dizemos que é.
3. **`Grupo`/`Lote`/`Empresa` mudam de tipo entre os anos** (texto x número) — forçados a `str`,
   senão o parquet falha na conversão.
4. **`Linha` é `"200210 - TERM ..."`**, código de 6 caracteres, enquanto `linha_blt` da bilhetagem
   usa `"NNNN-NN"`. A conversão é inserir o hífen na posição 4 — inclusive nas linhas noturnas
   (`N40211` → `N402-11`), conferido na Seção 4.

Requer `xlrd` (para `.xls` legado) e `openpyxl` (para `.xlsx`), ambos em `requirements.txt`.

In [3]:
COLS_OFICIAL = {
    "data": "Data", "grupo": "Grupo", "lote": "Lote", "empresa": "Empresa", "linha": "Linha",
    "passageirospagtesemdinheiro": "of_dinheiro",
    "passageiroscomumevt": "of_comum_vt",
    "passageirospgtsbucomumm": "of_bu_comum_mensal",
    "passageirospagtesestudante": "of_estudante",
    "passageirospgtsbuestmensal": "of_bu_est_mensal",
    "passageirospgtsbuvtmensal": "of_bu_vt_mensal",
    "passageirospagantes": "of_pagantes",
    "passageirosintonibusonibus": "of_integrados",
    "passageiroscomgratuidade": "of_gratuidade",
    "passageiroscomgratuidadeest": "of_gratuidade_est",
    "totpassageirostransportados": "of_total",
}
COLS_NUM = [c for c in COLS_OFICIAL.values() if c.startswith("of_")]


def _norm(s):
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode()
    return re.sub(r"[^a-z]", "", s.lower())


def le_oficial_um(caminho):
    bruto = pd.read_excel(caminho, sheet_name=0, header=None)
    linha_cab = next(i for i in range(min(12, len(bruto)))
                     if {"data", "linha"} <= {_norm(v) for v in bruto.iloc[i]})

    df = bruto.iloc[linha_cab + 1:].copy()
    df.columns = [COLS_OFICIAL.get(_norm(c), _norm(c)) for c in bruto.iloc[linha_cab]]
    df = df.loc[:, [c for c in df.columns if c in COLS_OFICIAL.values()]]
    df = df[df["Data"].notna() & df["Linha"].notna()]
    df = df[df["Linha"].astype(str).str.strip() != "Linha"]   # rodapés que repetem o cabeçalho

    for c in df.columns:
        if c in COLS_NUM:
            df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype("int64")
        elif c != "Data":
            df[c] = df[c].astype(str).str.strip()

    df["cod_linha"] = df["Linha"].astype(str).str.strip().str.extract(r"^(\S+)")[0]
    df["linha_blt"] = df["cod_linha"].str.replace(r"^(.{4})(.{2})$", r"\1-\2", regex=True)

    df["data_conteudo"] = pd.to_datetime(df["Data"], dayfirst=True,
                                         errors="coerce").dt.strftime("%Y%m%d")
    df = df.drop(columns=["Data"])
    df["data"] = os.path.basename(caminho)[:8]
    return df

### Integridade dos arquivos brutos

Antes de compilar: cada um dos 731 arquivos abre e tem a linha de cabeçalho `Data`/`Linha`?
É esta checagem que pegou o `20230529.xls` corrompido na origem (ver Seção 4). Leva ~2-3 min.

Quando algum arquivo falha, a rota de recuperação que funcionou foi **abrir no Excel e salvar como
`.xlsx`** (o Excel tolera o índice OLE inconsistente que derruba o `xlrd`), renomeando o `.xls`
original para fora do `glob`. `repara_via_excel()` faz isso via COM — exige `pywin32`
(`pip install pywin32`, deliberadamente **fora** do `requirements.txt`, já que só serve para este
conserto pontual) e Excel instalado. Em máquina sem Excel,
`libreoffice --headless --convert-to xlsx` resolve o mesmo caso.

In [4]:
def arquivos_brutos():
    return sorted(glob.glob(os.path.join(PASTA_OFICIAL, "*.xls"))
                  + glob.glob(os.path.join(PASTA_OFICIAL, "*.xlsx")))


def checa_integridade(arquivos):
    """[(arquivo, motivo)] dos que nao abrem ou nao tem cabecalho Data/Linha."""
    ruins = []
    for a in arquivos:
        try:
            cab = pd.read_excel(a, sheet_name=0, header=None, nrows=12)
            if not any({"data", "linha"} <= {_norm(v) for v in cab.iloc[i]}
                       for i in range(len(cab))):
                ruins.append((a, "sem linha de cabecalho Data/Linha"))
        except Exception as e:
            ruins.append((a, f"{type(e).__name__}: {str(e)[:80]}"))
    return ruins


def repara_via_excel(caminho):
    """Reabre um .xls quebrado no Excel e salva como .xlsx; move o original para fora do glob.

    O Excel tolera o indice OLE inconsistente que faz o xlrd abortar. Retorna o caminho
    do .xlsx gerado, ou None se nao houver Excel (ou se ele tambem nao abrir o arquivo).
    """
    import win32com.client  # pywin32; so importado quando ha algo a reparar

    destino = os.path.splitext(caminho)[0] + ".xlsx"
    xl = win32com.client.Dispatch("Excel.Application")
    xl.Visible = False
    xl.DisplayAlerts = False
    try:
        wb = xl.Workbooks.Open(caminho)
        wb.SaveAs(destino, 51)          # 51 = xlOpenXMLWorkbook
        wb.Close(False)
    finally:
        xl.Quit()
    os.rename(caminho, caminho + ".corrompido_origem")   # fora do glob: nao duplica o dia
    return destino


ruins = checa_integridade(arquivos_brutos())
print(f"{len(arquivos_brutos())} arquivos | ilegiveis: {len(ruins)}")
for a, motivo in ruins:
    print("   ", os.path.basename(a), "|", motivo)

# Tentativa de reparo automatico. Deixada explicitamente sob flag: mexe na pasta de dados
# brutos (renomeia o original), entao nao roda sozinha numa reexecucao rotineira.
REPARAR = False
if REPARAR:
    for a, _ in ruins:
        if a.endswith(".xls"):
            print("reparado ->", repara_via_excel(a))

731 arquivos | ilegiveis: 0


## Seção 3 — Compilação

In [5]:
if not FORCE_RECOMPUTE and os.path.exists(CAMINHO_SAIDA):
    oficial = pd.read_parquet(CAMINHO_SAIDA)
    print(f"Carregado do cache: {CAMINHO_SAIDA} ({len(oficial):,} linhas). "
          f"Defina FORCE_RECOMPUTE=True para recompilar.")
else:
    arquivos = arquivos_brutos()
    print(f"{len(arquivos)} arquivos a compilar", flush=True)

    partes, erros = [], []
    for i, a in enumerate(arquivos, 1):
        try:
            partes.append(le_oficial_um(a))
        except Exception as e:
            erros.append((os.path.basename(a), f"{type(e).__name__}: {str(e)[:70]}"))
        if i % 100 == 0:
            print(f"  {i}/{len(arquivos)}", flush=True)

    oficial = pd.concat(partes, ignore_index=True)

    dt = pd.to_datetime(oficial["data"], format="%Y%m%d")
    oficial["Ano"] = oficial["data"].str[:4]
    oficial["Mes"] = oficial["data"].str[4:6]
    # tipo_dia junto da base evita recalcular o weekday em cada análise
    oficial["tipo_dia"] = dt.dt.dayofweek.map(lambda w: "sabado" if w == 5 else
                                              ("domingo" if w == 6 else "util"))

    oficial.to_parquet(CAMINHO_SAIDA, index=False)
    print(f"\nSalvo em {CAMINHO_SAIDA}")
    if erros:
        print(f"arquivos que falharam ({len(erros)}):")
        for a, m in erros:
            print("   ", a, m)

print(f"\n{len(oficial):,} linhas | {oficial['data'].nunique()} datas | "
      f"{oficial['linha_blt'].nunique()} linhas de ônibus")
oficial.head(3)

731 arquivos a compilar


  100/731


  200/731


  300/731


  400/731


  500/731


  600/731


  700/731



Salvo em ../outputs/01/Dados_4Meses_Domingos_2023_2024_site.parquet

989,311 linhas | 731 datas | 1471 linhas de ônibus


,Grupo,Lote,Empresa,Linha,of_dinheiro,of_bu_comum_mensal,of_estudante,of_bu_est_mensal,of_bu_vt_mensal,of_pagantes,...,of_gratuidade_est,of_total,cod_linha,linha_blt,data_conteudo,data,of_comum_vt,Ano,Mes,tipo_dia
0,GRUPO ARTICULAÇÃO,AR0,Ambiental,N40211 - METRO ITAQUERA/TERM VL CARRAO,2,1,1,0,0,25,...,1,39,N40211,N402-11,20230101,20230101,NaN,2023,01,domingo
1,GRUPO ARTICULAÇÃO,AR0,Ambiental,N40511 - TERM V CARRAO/METRO ITAQUERA,6,1,0,0,0,33,...,2,56,N40511,N405-11,20230101,20230101,NaN,2023,01,domingo
2,GRUPO ARTICULAÇÃO,AR0,Ambiental,200210 - TERM P D PEDRO II/TERM BANDEIR,23,15,0,0,0,132,...,0,471,200210,2002-10,20230101,20230101,NaN,2023,01,domingo


## Seção 4 — Validações

Duas anomalias conhecidas, ambas verificadas — a primeira **resolvida**, a segunda benigna e a
ser deixada em paz:

- **`20230529.xls` vinha corrompido da origem — recuperado via Excel.** O arquivo publicado tem
  header OLE2 válido, mas o índice interno (SAT) é inconsistente e o `xlrd` aborta com
  `AssertionError` mesmo com `ignore_workbook_corruption=True`. Re-baixar não resolve: a única
  versão no portal tem **exatamente os mesmos 527.360 bytes** do arquivo local, e os sufixos de
  republicação `(1)`/`(2)`/`(3)` retornam 404. O Excel, que tolera esse tipo de defeito, abre o
  arquivo normalmente (2 planilhas, 1.550 linhas); convertido para `20230529.xlsx`, o conteúdo é
  o esperado (`Data = 29/05/2023`, cabeçalho na terceira linha). O `.xls` original foi renomeado
  para `20230529.xls.corrompido_origem` — fora do `glob`, para o dia não entrar duas vezes na
  compilação. **Com isso a base fecha em 731/731 dias.** Se a pasta for reconstruída do zero em
  outra máquina o defeito reaparece: rode a célula de integridade da Seção 2 com `REPARAR = True`.
- **`PAESE` não é uma linha de ônibus.** Aparece como `cod_linha` de 5 caracteres (~1 registro por
  dia) e é o serviço emergencial que substitui o metrô em ocorrências. Fica na base com
  `linha_blt = "PAESE"`; não casa com `linha_blt` da bilhetagem, e é isso mesmo. Todos os demais
  registros têm código de 6 caracteres e convertem para `NNNN-NN`.

In [6]:
# 1. Cobertura: todos os dias dos anos pedidos?
esperadas = {f"{d:%Y%m%d}" for ano in ANOS for d in dias_do_ano(ano)}
obtidas = set(oficial["data"].unique())
print(f"datas esperadas={len(esperadas)} obtidas={len(obtidas)} faltando={len(esperadas - obtidas)}")
if esperadas - obtidas:
    print("  faltando:", sorted(esperadas - obtidas))

# 2. A data do conteúdo bate com a data do nome do arquivo?
div = (oficial["data"] != oficial["data_conteudo"]).sum()
print(f"registros com data do conteúdo != data do nome: {div}")

# 3. A identidade contábil das colunas fecha?
soma = (oficial["of_pagantes"] + oficial["of_integrados"]
        + oficial["of_gratuidade"] + oficial["of_gratuidade_est"])
print(f"linhas em que pagantes+integrados+gratuidades != total: {(soma != oficial['of_total']).sum()}")

# 4. A conversão de código de linha funcionou, inclusive nas noturnas?
noturnas = oficial.loc[oficial["cod_linha"].str.startswith("N"), ["cod_linha", "linha_blt"]]
print(f"\nlinhas noturnas: {noturnas['cod_linha'].nunique()} códigos distintos")
print(noturnas.drop_duplicates().head(3).to_string(index=False))

fora = oficial.loc[~oficial["linha_blt"].str.match(r"^.{4}-.{2}$")]
print(f"\nlinha_blt fora do formato NNNN-NN: {len(fora)} registros")
print("códigos:", sorted(fora["cod_linha"].unique()), "(esperado: só PAESE)")

datas esperadas=731 obtidas=731 faltando=0
registros com data do conteúdo != data do nome: 0
linhas em que pagantes+integrados+gratuidades != total: 0

linhas noturnas: 150 códigos distintos
cod_linha linha_blt
   N40211   N402-11
   N40511   N405-11
   N14311   N143-11

linha_blt fora do formato NNNN-NN: 91 registros
códigos: ['PAESE'] (esperado: só PAESE)


### Em 2024 a quebra por modalidade colapsa — e isso é a política, não erro de parse

Nos domingos de 2024, `of_pagantes` fica ~0 e praticamente tudo cai em `of_gratuidade`: sob a
Tarifa Zero ninguém paga no domingo. `of_integrados` também vai a zero, porque sem tarifa não há
integração a contabilizar.

**Não "conserte" isso.** A consequência prática é que a comparação por modalidade de pagamento só
faz sentido dentro de 2023; entre os anos, o único campo comparável é **`of_total`**, que continua
sendo a soma de todos os embarques nos dois regimes.

In [7]:
por_regime = (oficial.groupby(["Ano", "tipo_dia"])[
                  ["of_pagantes", "of_integrados", "of_gratuidade", "of_total"]].sum() / 1e6)
print("milhões de passageiros por ano x tipo_dia:")
print(por_regime.round(1).to_string())

milhões de passageiros por ano x tipo_dia:


               of_pagantes  of_integrados  of_gratuidade  of_total
Ano  tipo_dia                                                     
2023 domingo          56.8           25.0           23.3     109.2
     sabado          119.5           50.7           27.9     206.8
     util            947.1          421.8          208.8    1666.5
2024 domingo           0.0            0.0          151.1     151.1
     sabado          117.0           49.6           37.6     212.4
     util            956.2          418.1          296.1    1759.3


## Seção 5 — Recorte de análise

Os notebooks de `03_comparacoes/` trabalham com `MESES = ["04","05","09","10"]`. O recorte é um
filtro sobre esta base, não um limite dela — o resto do ano continua disponível como contexto.

In [8]:
MESES_ANALISE = ["04", "05", "09", "10"]
recorte = oficial[oficial["Mes"].isin(MESES_ANALISE)]

print(f"{len(recorte):,} linhas no recorte | {recorte['data'].nunique()} datas")
print("\ndias por ano x tipo_dia:")
print(recorte.groupby(["Ano", "tipo_dia"])["data"].nunique().unstack().to_string())

print("\nmédia de passageiros por dia (milhões):")
por_dia = recorte.groupby(["Ano", "tipo_dia", "data"])["of_total"].sum().reset_index()
print((por_dia.groupby(["Ano", "tipo_dia"])["of_total"].mean() / 1e6).round(2).unstack().to_string())

331,657 linhas no recorte | 244 datas

dias por ano x tipo_dia:
tipo_dia  domingo  sabado  util
Ano                            
2023           18      18    86
2024           17      16    89

média de passageiros por dia (milhões):
tipo_dia  domingo  sabado  util
Ano                            
2023         2.08    4.01  6.59
2024         2.96    4.12  7.07
